In [0]:
import java.time.LocalDate
import java.time.LocalDateTime
import org.apache.spark.sql.{SparkSession, Row}
import org.apache.spark.sql.types._
import scala.util.Try
import scala.collection.JavaConverters._
import org.apache.spark.sql.functions.col
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types.DateType
import java.time.format.DateTimeFormatter
import org.apache.spark.sql.{DataFrame, Row}
import org.apache.spark.sql.types.{StructType, StructField, StringType}
import org.apache.spark.sql.Column
import org.apache.spark.sql.functions._
import java.time._
import java.time.{Instant, ZoneId, ZonedDateTime}

In [0]:
// Get the current Instant
val now: Instant = Instant.now()

// Convert Instant to ZonedDateTime in a specific time zone (e.g., UTC or your local zone)
val zonedDateTime: ZonedDateTime = now.atZone(ZoneId.of("UTC"))

// Define a formatter with your desired pattern
val formatter: DateTimeFormatter = DateTimeFormatter.ofPattern("yyyy-MM-dd")

// Format the ZonedDateTime
val formattedDate: String = zonedDateTime.format(formatter)
val currentDateUTC = formattedDate

// Extraer año, mes y día como enteros
val yearCurrent = zonedDateTime.getYear
val monthCurrent = f"${zonedDateTime.getMonthValue}%02d"
val dayCurrent = f"${zonedDateTime.getDayOfMonth}%02d"

val basePathLanding = dbutils.widgets.get("path_Landing")

// Construcción de rutas
val pathLandingFinal = s"$basePathLanding/$yearCurrent/$monthCurrent/$dayCurrent"




In [0]:
val spark = SparkSession.builder().appName("caidasIndividuales").getOrCreate()

// Esquema del DataFrame
val schema = StructType(Seq(
  StructField("ACCESSID", StringType, true),
  StructField("ESTADOPPOE", StringType, true),
  StructField("ESTADOINVENTARIO", StringType, true),
  StructField("AGENCIA", StringType, true),
  StructField("ULTIMORADIUSINTERIM", StringType, true),
  StructField("ULTIMORADIUSSTART", StringType, true),
  StructField("ULTIMORADIUSSTOP", StringType, true),
  StructField("NOMBREARCHIVO", StringType, true),
  StructField("FILENAME_SPARK", StringType, true),
  StructField("TS", StringType, true)
))

// Función para verificar si la ruta tiene archivos
def checkPath(path: String): Boolean = Try(dbutils.fs.ls(path)).getOrElse(Seq()).nonEmpty

// Lista vacía de Row para crear DataFrames vacíos
val emptyList: java.util.List[Row] = Seq.empty[Row].asJava

// Obtener nombres de las columnas del esquema
val selectedColumns = schema.fieldNames.map(col)

// Verificar si las rutas tienen datos y cargar los DataFrames
val dfr1 = if (checkPath(pathLandingFinal)) {
  println(s"La ruta $pathLandingFinal tiene archivos.")
  spark.read.format("csv").option("delimiter", ";").option("header", "true").option("inferSchema", "false").schema(schema).load(pathLandingFinal)
} else {
    spark.createDataFrame(emptyList, schema)
}

In [0]:
//Rutas de Trabajo Descarga y preprocesamiento
var parsing_in = dbutils.widgets.get("parsing_in_folder")
var parsing_out = dbutils.widgets.get("parsing_out_folder")

// Función para limpiar columnas String
def cleanStringColumn(colName: String): Column = {
  val trimmedColumn = trim(col(colName))
  when(trimmedColumn === "" || trimmedColumn === "null" || col(colName).isNull, null)
    .otherwise(trimmedColumn)
    .alias(colName)
}

try {
  val TimestampPattern = "yyyy-MM-dd HH:mm:ss"

// Se leen todos los archivos que existen en la ruta de entrada que han sido extraídos desde el origen.
  val df1 = spark.read.option("header", "true").option("inferSchema", "false").option("delimiter", ";").schema(schema).csv(parsing_in)
//Eliminan Dupliados
  val df2 = df1.dropDuplicates()

  // Realiza Cruce entre  data de entrada y data Landing para agregar solo registros nueuvos.
  //val df_final = df2.join(dfr1, usingColumns=Seq("ACCESSID","ESTADOPPOE","ESTADOINVENTARIO","AGENCIA","ULTIMORADIUSINTERIM","ULTIMORADIUSSTART","ULTIMORADIUSSTOP","FILENAME_SPARK","TS"), //joinType="left_anti")
  val df_final = df2.union(dfr1)

// Construir el nombre del archivo, utilizamos misma fecha UTC Instant.now()
val currentDate = zonedDateTime
val formatter2 = DateTimeFormatter.ofPattern("yyyyMMdd")
val currentDateStr = currentDate.format(formatter2)
val filename = s"CAIDAS_INDIVIDUALES_$currentDateStr.csv"

  // Guarda el archivo en la carpeta de salida
  df_final.repartition(1).write.option("header", "true").option("delimiter", ";").mode("append").csv(parsing_out)

// Renombrar el archivo guardado con el filename construido
val files = dbutils.fs.ls(parsing_out)
val oldFilePathOption = files.find(file => file.name.startsWith("part-"))

oldFilePathOption match {
  case Some(oldFile) =>
    val oldFilePath = oldFile.path
    val newFilePath = parsing_out + filename
    dbutils.fs.mv(oldFilePath, newFilePath)
    println(s"El archivo ha sido renombrado a: $filename")
  case None =>
    println("No se encontró ningún archivo que cumpla con el criterio.")
}
} catch {
  case e: Exception =>
    println("[ERROR] " + e)
    throw e
}

In [0]:
val files = dbutils.fs.ls(parsing_in)

//Elimina archivos y carpetas temporales.
def deleteFilesAndFolders(path: String): Unit = {
  val filesAndDirs = dbutils.fs.ls(path)

  filesAndDirs.foreach(file => {
    if (file.isFile) {
      dbutils.fs.rm(file.path, true)
    } 
    else if (file.isDir) {
      deleteFilesAndFolders(file.path)
    }
  })
  
  dbutils.fs.rm(path, true)
}

deleteFilesAndFolders(parsing_in)